# RRIvis Basic Usage Tutorial

This notebook demonstrates the basic usage of RRIvis for simulating radio interferometry visibilities.

## What You'll Learn

1. How to check available backends (CPU/GPU)
2. How to create a Simulator from code or config file
3. How to run a visibility simulation
4. How to visualize and save results

## 1. Setup and Import

In [ ]:
# Import RRIvis
from pathlib import Path

# Standard imports
import rrivis
from rrivis import Simulator
from rrivis.backends import get_backend, list_backends
from rrivis.simulator import list_simulators

print(f"RRIvis version: {rrivis.__version__}")

## 2. Check Available Backends

RRIvis supports multiple computation backends:
- **numpy**: CPU baseline (always available)
- **jax**: GPU/TPU acceleration (NVIDIA, AMD, Apple Silicon)
- **numba**: Production backend with Dask distributed support

In [ ]:
# Check available backends
backends = list_backends()
print("Available backends:")
for name, available in backends.items():
    status = "available" if available else "not available"
    print(f"  {name}: {status}")

# Check available simulators
simulators = list_simulators()
print("\nAvailable simulators:")
for name, description in simulators.items():
    print(f"  {name}: {description}")

## 3. Create a Simple Simulation

### Option A: Programmatic Configuration

Create a simulator by specifying parameters directly in Python:

In [ ]:
# Get path to example antenna layout
# Adjust this path based on your installation
repo_root = Path("../..").resolve()
antenna_file = repo_root / "antenna_layout_examples" / "example_rrivis_format.txt"

print(f"Antenna file exists: {antenna_file.exists()}")
if antenna_file.exists():
    print(f"Using antenna layout: {antenna_file}")

In [ ]:
# Create simulator with programmatic configuration
sim = Simulator(
    config={
        "antenna_layout": {
            "antenna_positions_file": str(antenna_file)
            if antenna_file.exists()
            else None,
            "antenna_file_format": "rrivis",
            "all_antenna_diameter": 14.0,
        },
        "obs_frequency": {
            "frequencies_hz": [100e6, 120e6, 140e6, 160e6, 180e6, 200e6],
            "frequency_unit": "MHz",
        },
        "sky_model": {"sources": [{"kind": "test_sources"}]},
        "visibility": {"sky_representation": "point_sources"},
        "location": {
            "lat": -30.72,  # HERA latitude (degrees)
            "lon": 21.43,  # HERA longitude (degrees)
            "height": 1073.0,  # meters
        },
        "obs_time": {"start_time": "2025-01-15T00:00:00"},
    },
    backend="auto",  # Auto-detect best backend
)

print(f"Simulator created with backend: {sim._backend_name}")

### Option B: From Configuration File

Alternatively, load configuration from a YAML file:

In [ ]:
# Example: Load from config file (uncomment to use)
# config_file = repo_root / "configs" / "01_parabolic_10db_taper.yaml"
# sim = Simulator.from_config(str(config_file))

## 4. Setup and Inspect

The `setup()` method loads antennas, generates baselines, and prepares sources:

In [ ]:
# Setup the simulation
sim.setup()

# Inspect the setup
print("Simulation Setup:")
print(f"  Antennas: {len(sim._antennas) if sim._antennas else 'N/A'}")
print(f"  Baselines: {len(sim._baselines) if sim._baselines else 'N/A'}")
print(f"  Sources: {len(sim._sources) if sim._sources else 'N/A'}")
print(
    f"  Frequencies: {len(sim._frequencies_hz) if sim._frequencies_hz is not None else 'N/A'}"
)

# Estimate memory usage
memory_mb = sim.get_memory_estimate()
print(f"\nEstimated memory usage: {memory_mb:.2f} MB")

## 5. Run the Simulation

Execute the visibility calculation using the RIME algorithm:

In [ ]:
# Run the simulation with progress bar
results = sim.run(progress=True)

print("\nSimulation complete!")

## 6. Explore Results

The results dictionary contains visibilities and metadata:

In [ ]:
# Check what's in the results
if results:
    print("Results keys:", list(results.keys()))

    # Get visibility data
    if "visibilities" in results:
        vis = results["visibilities"]
        print(f"\nNumber of baselines: {len(vis)}")

        # Look at a sample baseline
        sample_key = list(vis.keys())[0]
        sample_vis = vis[sample_key]
        print(f"\nSample baseline: {sample_key}")
        print(f"  Shape: {sample_vis.shape}")
        print(f"  Dtype: {sample_vis.dtype}")
        print("  Sample values (first 3 frequencies):")
        print(f"    {sample_vis[:3]}")

## 7. Visualize Results

Generate plots of antenna layout and visibilities:

In [ ]:
# Generate plots
try:
    sim.plot(plot_type="all")
except Exception as e:
    print(f"Plotting not available: {e}")
    print("Try running in a different environment or check plotting dependencies.")

## 8. Save Results

Save the simulation results to disk:

In [ ]:
# Save results to HDF5 format
output_dir = Path("simulation_output")
sim.save(str(output_dir), format="hdf5")
print(f"Results saved to: {output_dir.resolve()}")

# List saved files
if output_dir.exists():
    print("\nSaved files:")
    for f in output_dir.iterdir():
        print(f"  {f.name}")

## 9. GPU Acceleration (Optional)

If you have a GPU available, you can use the JAX backend for significant speedup:

In [ ]:
# Check if GPU is available
backends = list_backends()

if backends.get("jax", False):
    print("JAX backend available!")

    # Get backend info
    try:
        jax_backend = get_backend("jax")
        device_info = jax_backend.get_device_info()
        print(f"Device info: {device_info}")
    except Exception as e:
        print(f"Could not get JAX device info: {e}")
else:
    print("JAX backend not available.")
    print("Install with: pip install rrivis[gpu]")

## 10. Benchmark (Optional)

Compare performance between backends:

In [ ]:
# Benchmark the simulation (runs multiple times)
benchmark_results = sim.benchmark(n_runs=3)

print("Benchmark Results:")
print(f"  Backend: {benchmark_results.get('backend', 'N/A')}")
print(f"  Mean time: {benchmark_results.get('mean_time', 0):.3f} seconds")
print(f"  Std dev: {benchmark_results.get('std_time', 0):.3f} seconds")

## Summary

In this notebook, you learned how to:

1. **Check backends**: Use `list_backends()` to see available computation backends
2. **Create a Simulator**: Either programmatically or from a YAML config file
3. **Setup and run**: Use `setup()` and `run()` to execute the simulation
4. **Visualize**: Use `plot()` to generate visualizations
5. **Save results**: Use `save()` to export data in HDF5 or other formats

### Next Steps

- Explore different sky models (GLEAM, GSM)
- Try different antenna configurations
- Use GPU acceleration for larger simulations
- Explore the Jones matrix framework for instrumental effects

### Documentation

For more information, see the full documentation at: https://rrivis.readthedocs.io/